# Suite MUT — Mutations: the schemas of the verbs a transaction applies

`Mutations` holds the schemas of the mutations a transaction applies
([CAP-129: journals replayed by execution](../../../mbse-basis/docs/capabilities/CAP-129.md#transactions-on-the-wire)): an object's verbs (`OfObject`, its properties' and its adjacencies'), an adjacency's
(`OfAdjacency`), value verbs nested in them, the nouns `Ref`, `Def` and `Slice`, and a pattern's spec, `OfPattern`.
Mutations are schema-governed: every store holds their schemas, and mutations are built and written as any value is.
Every store also holds the basic scalars, `Schemas.OfBool` to `Schemas.OfBytes`.

As in Python, with the language's differences (docs/EQUIVALENCE.md): builders and the store's paths are `Proxy`
objects typed loosely, so callbacks are annotated `(c: any)`; ints are `bigint`s, and plain mappings are `Map`s
(compared with `equal` against `map(...)`).

In [ ]:
import { Builders as B, JSON, Plain } from "@mbse/schemas/Framework";
import { assert, equal, map, raises, same } from "./support.js";

const store: any = new B.Store();
const M = { // the store's own mutation schemas, through its paths
  OfPattern: store.Mutations.OfPattern.as_schema(), OfObject: store.Mutations.OfObject.as_schema(),
  OfAdjacency: store.Mutations.OfAdjacency.as_schema(),
  OfAny: store.Mutations.OfAny.as_schema(), OfContainer: store.Mutations.OfContainer.as_schema(),
  OfLink: store.Mutations.OfLink.as_schema(), PropertySet: store.Mutations.OfProperty.Set.as_schema(),
  AnySet: store.Mutations.OfAny.Set.as_schema(), ScalarSet: store.Mutations.OfScalar.Set.as_schema(),
  SCHEMAS: store.names().filter((name: string) => name.startsWith("Mutations.")).map((name: string) => store.registered(name)),
};

/** A path as plain data: from `singleton`, then, for a schema object's, through each module and then the schema by name
 * (the root module is the singleton "."). */
function writtenAt(singleton: string, ...names: string[]): any {
  let nested: any = { target: {} };
  for (const name of [...names].reverse()) nested = { entry: { i: "schemas", o: "", properties: { name }, next: nested } };
  return { singleton, next: nested };
}

function plain(value: unknown, schema: unknown = M.OfPattern): any {
  return Plain.ToPlain(store)(schema, value);
}

/** The keys of a plain mapping, as Python's `list(mapping)`. */
const keys = (mapping: Map<string, unknown>) => [...mapping.keys()];

## MUT-01 · The mutation schemas, valid, and the basic scalars, held by every store

In [ ]:
{
  assert(M.SCHEMAS.every((schema: any) => schema.validate().length === 0) && M.SCHEMAS.length === 65); // a spec's, mutation expressions' (TXN), a transaction's session and where (PRV)
  for (const schema of M.SCHEMAS) {
    assert(store.registered(schema.name) === schema);
  }
  const other: any = new B.Store();
  assert(other.registered("Mutations.OfPattern") !== M.OfPattern && other.Mutations.OfPattern.as_schema().validate().length === 0); // each its own
  for (const name of ["OfBool", "OfInt", "OfFloat", "OfStr", "OfBytes"] as const) {
    const scalar = store.Schemas[name].as_schema();
    assert(scalar.name === `Schemas.${name}` && store.registered(scalar.name) === scalar);
  }
  assert(store.Schemas.OfStr.as_schema().type === String && store.Schemas.OfObject.as_schema().properties.get("ref")?.schema === store.Schemas.OfBool.as_schema());
  assert(same(M.OfObject.branches.map((b: any) => b.name), ["as_property", "as_adjacency", "spec"])); // a pattern's own spec too
  assert(same(M.OfAdjacency.branches.map((b: any) => b.name), ["insert", "remove", "clear"]));
  assert(same(M.OfAny.branches.map((b: any) => b.name), ["set", "clear", "as_scalar", "as_container", "as_object", "as_union",
    "as_intersection"]));
  assert(M.PropertySet !== M.AnySet && M.AnySet !== M.ScalarSet); // each family its own
}

## MUT-02 · An object's verbs: a property set, and changed through the nested value verbs; an object defined where it is linked

In [ ]:
{
  const aged = store.Mutations.OfObject((u: any) => u.as_property((p: any) => (
    p
      .name("age")
      .mutations([store.Mutations.OfProperty((v: any) => v.set((s: any) => s.value(37n)))])
  )));
  const deep = store.Mutations.OfObject((u: any) => u.as_property((p: any) => (
    p
      .name("address")
      .mutations([store.Mutations.OfProperty((v: any) => v.as_any((a: any) => a.mutations([
        store.Mutations.OfAny((x: any) => x.as_object((o: any) => o.mutations([
          store.Mutations.OfObject.AsProperty((q: any) => q.name("city").mutations([
            store.Mutations.OfProperty((w: any) => w.set((s: any) => s.value("Paris"))),
          ])),
        ]))),
      ])))])
  )));
  const defined = store.Mutations.OfLink((k: any) => k.set((s: any) => s.def((d: any) => (
    d
      .schema("crm.Address")
      .mutations([store.Mutations.OfObject((u: any) => u.as_property((p: any) => p.name("city").mutations([
        store.Mutations.OfProperty((v: any) => v.set((s: any) => s.value("Paris")))])))])
  ))));
  const written = plain(store.Mutations.OfPattern((c: any) => [c.at("ada").as_container((l: any) => [
    l.at(0n).as_union(aged), l.at(1n).as_union(deep)])]));
  assert(equal(written.get("ada")[0], map({ as_property: { name: "age", mutations: [{ set: { value: { int: 37n } } }] } })));
  const nested = written.get("ada")[1].get("as_property").get("mutations")[0].get("as_any").get("mutations")[0].get("as_object")
    .get("mutations")[0];
  assert(equal(nested, map({ name: "city", mutations: [{ set: { value: { str: "Paris" } } }] })));
  assert(equal(plain(defined, M.OfLink), map({ set: { def: { schema: "crm.Address", mutations: [
    { as_property: { name: "city", mutations: [{ set: { value: { str: "Paris" } } }] } }] } } })));
}

## MUT-03 · Value verbs: containers by slices, by count, and item by item in modulo order; scalars; unions and intersections

In [ ]:
{
  const items = store.Mutations.OfContainer((c: any) => c.as_items((i: any) => i.mutations([
    store.Mutations.OfAny((a: any) => a.as_scalar((s: any) => s.mutations([
      store.Mutations.OfScalar((x: any) => x.set((v: any) => v.value(0n))),
    ]))),
    store.Mutations.OfAny((a: any) => a.clear(store.Mutations.OfAny.Clear((c: any) => c))),
  ])));
  const edits = [
    store.Mutations.OfContainer((c: any) => c.insert((i: any) => i.slice((p: any) => p.as_object((s: any) => s.index(0n).items(["a", "b"]))))),
    store.Mutations.OfContainer((c: any) => c.remove((r: any) => r.index(1n).count(2n))),
    store.Mutations.OfContainer((c: any) => c.append((a: any) => a.items(["z"]))),
    store.Mutations.OfContainer((c: any) => c.prepend((a: any) => a.items(["y"]))),
    store.Mutations.OfContainer((c: any) => c.clear(store.Mutations.OfContainer.Clear((c: any) => c))),
    items,
  ];
  const held = edits.map((edit) => plain(edit, M.OfContainer));
  assert(equal(held.map(keys), [["insert"], ["remove"], ["append"], ["prepend"], ["clear"], ["as_items"]]));
  assert(equal(held[1], map({ remove: { index: { int: 1n }, count: 2n } })) && equal(held[4], map({ clear: {} })));
  assert(equal(held[5].get("as_items").get("mutations")[1], map({ clear: {} })));
  const branch = store.Mutations.OfAny((a: any) => a.as_union((u: any) => u.mutations([
    store.Mutations.OfUnion.AsBranch((b: any) => b.name("circle").mutations([
      store.Mutations.OfProperty((p: any) => p.clear(store.Mutations.OfProperty.Clear((c: any) => c))),
    ])),
  ])));
  const part = store.Mutations.OfAny((a: any) => a.as_intersection((i: any) => i.mutations([
    store.Mutations.OfIntersection.AsPart((b: any) => b.name("stamp").mutations([])),
  ])));
  assert(equal(plain(branch, M.OfAny), map({ as_union: { mutations: [{ name: "circle", mutations: [{ clear: {} }] }] } })));
  assert(equal(plain(part, M.OfAny), map({ as_intersection: { mutations: [{ name: "stamp", mutations: [] }] } })));
}

## MUT-04 · Adjacencies: entries inserted and removed with their links and properties; a link referring by path or by symbol, or cleared

In [ ]:
{
  const bob = store.Mutations.OfObject.Ref((r: any) => r.symbol("bob"));
  const knows = store.Mutations.OfObject((u: any) => u.as_adjacency((a: any) => a.name("acquaintances").mutations([
    store.Mutations.OfAdjacency((j: any) => j.insert((i: any) => i.entries([
      store.Mutations.OfEntry((e: any) => e.mutations([
        store.Mutations.OfEntry.Member((m: any) => m.as_link((l: any) => l.name("known").mutations([
          store.Mutations.OfLink((k: any) => k.set((s: any) => s.ref((f: any) => f.path((p: any) => ( // by its path
            p.singleton("crm.Bob").next((n: any) => n.target((t: any) => t))))))),
        ]))),
        store.Mutations.OfEntry.Member((m: any) => m.as_property((p: any) => p.name("since").mutations([
          store.Mutations.OfProperty((v: any) => v.set((s: any) => s.value(2020n))),
        ]))),
      ])),
      store.Mutations.OfEntry((e: any) => e.mutations([
        store.Mutations.OfEntry.Member((m: any) => m.as_link((l: any) => l.name("known").mutations([
          store.Mutations.OfLink((k: any) => k.set((s: any) => s.ref(bob))), // by its symbol
        ]))),
        store.Mutations.OfEntry.Member((m: any) => m.as_property((p: any) => p.name("since").mutations([
          store.Mutations.OfProperty((v: any) => v.set((s: any) => s.value(2021n))),
        ]))),
      ])),
    ]))),
    store.Mutations.OfAdjacency((j: any) => j.remove((r: any) => r.entries([
      store.Mutations.OfEntry((e: any) => e.mutations([
        store.Mutations.OfEntry.Member((m: any) => m.as_link((l: any) => l.name("known").mutations([
          store.Mutations.OfLink((k: any) => k.set((s: any) => s.ref(bob))),
        ]))),
        store.Mutations.OfEntry.Member((m: any) => m.as_property((p: any) => p.name("since").mutations([
          store.Mutations.OfProperty((v: any) => v.set((s: any) => s.value(2021n))),
        ]))),
      ])),
    ]))),
    store.Mutations.OfAdjacency((j: any) => j.clear(store.Mutations.OfAdjacency.Clear((c: any) => c))),
  ])));
  const unlinked = store.Mutations.OfLink((k: any) => k.clear(store.Mutations.OfLink.Clear((c: any) => c)));
  const written = plain(store.Mutations.OfPattern((c: any) => [c.at("ada").as_container((l: any) => [
    l.at(0n).as_union(knows)])])).get("ada")[0].get("as_adjacency");
  assert(written.get("name") === "acquaintances" && equal(written.get("mutations").map(keys), [["insert"], ["remove"], ["clear"]]));
  assert(equal(written.get("mutations")[2], map({ clear: {} }))); // every entry it holds
  const [first, second] = written.get("mutations")[0].get("insert").get("entries");
  assert(equal(first.get("mutations")[0], map({ as_link: { name: "known", mutations: [{ set: { ref: { path: writtenAt("crm.Bob") } } }] } })));
  assert(equal(first.get("mutations")[1], map({ as_property: { name: "since", mutations: [{ set: { value: { int: 2020n } } }] } })));
  assert(equal(second.get("mutations")[0].get("as_link").get("mutations"), [map({ set: { ref: { symbol: "bob" } } })]));
  assert(equal(plain(unlinked, M.OfLink), map({ clear: {} })));
}

## MUT-05 · Mutations read back from plain data and from JSON, as any value is

In [ ]:
{
  const named = store.Mutations.OfObject((u: any) => u.as_property((p: any) => p.name("name").mutations([
    store.Mutations.OfProperty((v: any) => v.set((s: any) => s.value("Ada")))])));
  const written = plain(store.Mutations.OfPattern((c: any) => [c.at("ada").as_container((l: any) => [l.at(0n).as_union(named)])]));
  const back = Plain.FromPlain(store)(M.OfPattern, written);
  assert(equal(plain(back), written));
  const text = JSON.ToJSON(store)(M.OfPattern, back);
  assert(equal(JSON.loads(text), written) && equal(plain(JSON.FromJSON(store)(M.OfPattern, text)), written));
  raises(Error, () => Plain.FromPlain(store)(M.OfObject, map({ as_property: {}, as_adjacency: {} }))); // a union value holds one branch
}

## MUT-06 · A spec's verbs: an object's properties and adjacencies, an adjacency's entries inserted, each a `Mutations.OfPattern` by symbol; its payloads bare

In [ ]:
{
  /** crm.Person; crm.Book, a singleton holding its contacts. */
  const crm = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("crm/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Holds: S.OfRelation, Person: S.OfObject, Book: S.OfObject })
      .spec([
        (_: any) => _.Holds.name("crm.Holds").links([(t: any) => t.name("book"), (t: any) => t.name("contact")]),
        (_: any) => _.Person.name("crm.Person").ref(true).properties({ name: S.OfStr }),
        (_: any) => _.Book.name("crm.Book").ref(true).singleton("crm.TheBook").adjacencies({
          contacts: (t: any) => t.relation([_.Holds]).me("book") }),
      ])]))]);
  };
  /** crm.Filing, a pattern over what `crm` installed: Ada, one of the book's contacts. */
  const filing = (txn: any): any => txn.uid("filing/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Filing: txn.Schemas.OfPattern })
    .spec([(_: any) => _.Filing.name("crm.Filing").symbols({ book: txn.crm.Book, ada: txn.crm.Person }).spec([
      (_: any) => _.book.contacts([_.ada]), (_: any) => _.ada.name("Ada")])])]))]);
  const spec6: any = new B.Store();
  spec6.transaction(crm);
  spec6.transaction(filing);
  const pattern6 = spec6.registered("crm.Filing");
  const M6 = spec6.Mutations;
  const contact6 = M6.OfObject((m: any) => m.as_adjacency((a: any) => a.name("contacts").mutations([
    M6.OfAdjacency((j: any) => j.insert((i: any) => i.entries([M6.OfEntry((e: any) => e.mutations([
      M6.OfEntry.Member((x: any) => x.as_link((l: any) => l.name("contact").mutations([
        M6.OfLink((k: any) => k.set((s: any) => s.ref(M6.OfObject.Ref((r: any) => r.symbol("ada"))))),
      ]))),
    ]))])))])));
  const named6 = M6.OfObject((m: any) => m.as_property((p: any) => p.name("name").mutations([
    M6.OfProperty((v: any) => v.set((s: any) => s.value("Ada")))])));
  const built6 = M6.OfPattern((c: any) => [c.at("book").as_container((l: any) => [l.at(0n).as_union(contact6)]),
    c.at("ada").as_container((l: any) => [l.at(0n).as_union(named6)])]);
  const written6 = Plain.ToPlain(spec6)(M6.OfPattern, built6);
  assert(equal(Plain.ToPlain(spec6)(M6.OfPattern, Plain.FromPlain(spec6)(M6.OfPattern, written6)), written6)); // as any value
  const set6 = written6.get("ada")[0].get("as_property").get("mutations")[0].get("set");
  assert(equal(set6.get("value"), map({ str: "Ada" }))); // tagged, as any value is
  set6.set("value", "Ada"); // a spec's payload is bare, as it records it
  assert(equal(pattern6.spec, [map({ book: written6.get("book") }), map({ ada: written6.get("ada") })])); // what it records
  raises(Error, () => Plain.FromPlain(spec6)(M6.OfPattern, map({ ada: [{ as_property: {}, as_adjacency: {} }] }))); // one verb each
}